# OLMo 3.1 32B Think vs Instruct Subset Planner

Purpose: run a small matched candidate pool through `OLMo-3.1-32B-Instruct` and `OLMo-3.1-32B-Think` with vLLM on a Colab A100, then estimate how many examples fit into an approximately 24-hour run per checkpoint.

Default pool size is 18 examples: `3 digit lengths x 3 slices x 2 prompt modes`. With the current reference numbers, that should be roughly 16 minutes for Instruct at a 2k token cap and roughly 40 minutes for Think at a 4096-token thinking cap, excluding model load time.


## 1. Clone Or Update Repo

Edit `REPO_URL` if the GitHub remote is different or private. If the repo is private, authenticate with Colab/GitHub before running this cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/ndalton12/carry-trace.git'  # Change if needed.
BRANCH = 'main'
REPO_DIR = Path('/content/carry-trace')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Repo: {REPO_DIR}')
print(f'Commit: {commit}')
print(f'Python: {sys.version}')


## 2. Install Dependencies

This cell pins Torch and vLLM together. vLLM native extensions are compiled against specific Torch ABIs, so mixing Colab's preinstalled Torch with an arbitrary vLLM wheel can produce undefined-symbol import errors. After this cell finishes, restart the runtime once, then continue from the runtime setup cell.


In [ ]:
!python -m pip install -U pip uv
!python -m pip uninstall -y torch torchvision torchaudio xformers vllm vllm-flash-attn vllm-nccl-cu12 vllm-nccl-cu13 nvidia-cuda-runtime-cu13 nvidia-cublas-cu13 nvidia-cudnn-cu13 || true
!python -m pip install -e . --no-deps --ignore-requires-python
!python -m pip install 'accelerate>=1.0.0' 'datasets>=3.0.0' 'matplotlib>=3.9.0' 'numpy>=2.0.0' 'pandas>=2.2.0' 'pyarrow>=17.0.0' 'pydantic>=2.9.0' 'pyyaml>=6.0.2' 'rich>=13.8.0' 'seaborn>=0.13.2' 'transformers>=4.45.0' 'typer>=0.12.5' 'huggingface_hub[hf_transfer]>=0.23.0'
!python -m pip install --no-cache-dir --force-reinstall 'torch==2.8.0' 'torchvision==0.23.0' 'torchaudio==2.8.0' --index-url https://download.pytorch.org/whl/cu128
!python -m pip install --no-cache-dir --force-reinstall 'vllm==0.10.2'
print('Install complete. Restart the Colab runtime now, then continue from the runtime setup cell.')


## 3. Runtime Setup

Set `HF_TOKEN` in Colab secrets if Hugging Face auth is required. The benchmark writes local artifacts under `/content/carry-trace/runs/subset_planner`.


In [ ]:
import gc
import json
import math
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch

os.chdir(REPO_DIR)
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

import carry_trace
from carry_trace.config import DatasetConfig, GenerationParams, ModelSpec, RunnerConfig
from carry_trace.datasets import dump_dataset_row, generate_dataset
from carry_trace.io import ensure_dir, utc_now_iso, write_json, write_jsonl
from carry_trace.metrics import score_records, summarize_goal1
from carry_trace.models import make_runner

print('carry_trace imported from', carry_trace.__file__)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16 supported', torch.cuda.is_bf16_supported())

try:
    import vllm
    print('vllm', vllm.__version__)
except Exception as exc:
    print('vllm import failed:', type(exc).__name__, exc)
    raise

!nvidia-smi


## 4. Benchmark Configuration

`THINK_THINKING_TOKENS = 4096` means the first Think generation phase gets 4096 tokens, then `THINKING_FINAL_ANSWER_TOKENS` are reserved for the forced final-answer continuation if needed. The Think model therefore uses `max_new_tokens = 4096 + THINKING_FINAL_ANSWER_TOKENS`.


In [ ]:
BENCH_NAME = 'colab_olmo31_32b_think_instruct_subset_planner'
SEED = 20260611

INSTRUCT_MODEL_ID = 'allenai/Olmo-3.1-32B-Instruct'
THINK_MODEL_ID = 'allenai/Olmo-3.1-32B-Think'

RUN_INSTRUCT = True
RUN_THINK = True
BATCH_SIZE = 2
DTYPE = 'bfloat16'
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = None  # Set lower, e.g. 8192, if vLLM KV cache allocation is too high.

TARGET_HOURS_PER_CHECKPOINT = 24
REFERENCE_INSTRUCT_18_EXAMPLES_MINUTES = 16
REFERENCE_THINK_18_EXAMPLES_MINUTES = 40

EXAMPLES_PER_SLICE_PER_LENGTH = 1
DIGIT_LENGTHS = [4, 8, 12]
SLICES = ['no_carry', 'isolated_carry', 'long_carry_chain']
PROMPT_MODES = ['answer_only', 'free_cot']
DIGIT_FORMATS = ['standard']
ANSWER_FORMATS = ['standard']

INSTRUCT_MAX_NEW_TOKENS = 2048
THINK_THINKING_TOKENS = 4096
THINKING_FINAL_ANSWER_TOKENS = 100
THINK_MAX_NEW_TOKENS = THINK_THINKING_TOKENS + THINKING_FINAL_ANSWER_TOKENS

GENERATION_BASE = {
    'temperature': 0.6,
    'top_p': 0.95,
    'do_sample': True,
    'force_close_thinking': True,
    'thinking_final_answer_tokens': THINKING_FINAL_ANSWER_TOKENS,
}

RUN_ROOT = ensure_dir(REPO_DIR / 'runs' / 'subset_planner' / f'{BENCH_NAME}-{utc_now_iso().replace(":", "").replace("+", "Z")}')
print('run root:', RUN_ROOT)


## 5. Generate Candidate Pool

The pool is intentionally small so the whole notebook should fit in roughly an hour on one A100, assuming the 32B Think run is about 2.5x the Instruct runtime.


In [ ]:
dataset_config = DatasetConfig(
    name=BENCH_NAME,
    seed=SEED,
    output_dir=REPO_DIR / 'data' / 'generated',
    write_parquet=False,
    splits={'subset_probe': {'examples_per_slice_per_length': EXAMPLES_PER_SLICE_PER_LENGTH}},
    digit_lengths=DIGIT_LENGTHS,
    slices=SLICES,
    prompt_modes=PROMPT_MODES,
    digit_formats=DIGIT_FORMATS,
    answer_formats=ANSWER_FORMATS,
)
jsonl_path, manifest_path, examples = generate_dataset(dataset_config)
example_rows = [dump_dataset_row(example) for example in examples]
write_jsonl(RUN_ROOT / 'dataset.jsonl', example_rows)

print('dataset:', jsonl_path)
print('manifest:', manifest_path)
print('examples:', len(examples))
display(pd.DataFrame(example_rows)[['id', 'n_digits', 'slice_name', 'prompt_mode', 'a', 'b', 'answer']])


## 6. Reference Runtime Estimate

This is the prior estimate before running the models. The measured estimate after the runs appears later and should be used for final planning.


In [ ]:
def reference_capacity(minutes_for_18: float) -> dict[str, float]:
    seconds_per_example = minutes_for_18 * 60 / 18
    examples_per_24h = TARGET_HOURS_PER_CHECKPOINT * 3600 / seconds_per_example
    return {
        'seconds_per_example': seconds_per_example,
        'examples_per_hour': 3600 / seconds_per_example,
        'examples_per_24h': examples_per_24h,
    }

num_cells = len(DIGIT_LENGTHS) * len(SLICES) * len(PROMPT_MODES)
reference_rows = []
for name, minutes in [
    ('instruct_reference', REFERENCE_INSTRUCT_18_EXAMPLES_MINUTES),
    ('think_reference', REFERENCE_THINK_18_EXAMPLES_MINUTES),
]:
    row = {'model': name, **reference_capacity(minutes)}
    row['examples_per_cell_24h'] = math.floor(row['examples_per_24h'] / num_cells)
    reference_rows.append(row)

reference_df = pd.DataFrame(reference_rows)
paired_examples_per_cell = int(reference_df['examples_per_cell_24h'].min())
display(reference_df)
print(f'Cells: {num_cells}')
print(f'Paired subset if both checkpoints must finish within 24h: {paired_examples_per_cell} examples per cell, {paired_examples_per_cell * num_cells} total examples.')


## 7. Runner Helpers With Progress Reporting

Progress prints after each generated record. Because `batch_size=2`, records arrive after each vLLM batch completes.


In [ ]:
def generation_params(max_new_tokens: int) -> GenerationParams:
    return GenerationParams(max_new_tokens=max_new_tokens, **GENERATION_BASE)


def runner_config() -> RunnerConfig:
    kwargs = {
        'kind': 'vllm',
        'dtype': DTYPE,
        'batch_size': BATCH_SIZE,
        'trust_remote_code': False,
        'quantization': 'none',
        'tensor_parallel_size': 1,
        'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
    }
    if MAX_MODEL_LEN is not None:
        kwargs['max_model_len'] = MAX_MODEL_LEN
    return RunnerConfig(**kwargs)


def cleanup_vllm_runner(runner: object | None) -> None:
    if runner is not None:
        try:
            del runner.llm
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel
        destroy_model_parallel()
    except Exception as exc:
        print('vLLM cleanup note:', type(exc).__name__, exc)


def run_one_model(model_name: str, model_id: str, max_new_tokens: int) -> tuple[list[dict], list[dict], dict]:
    print(f'\n=== Loading {model_name}: {model_id} ===', flush=True)
    gen = generation_params(max_new_tokens)
    runner = make_runner(
        ModelSpec(name=model_name, model_id=model_id),
        runner_config(),
        gen,
    )
    records = []
    started = time.perf_counter()
    last_print = started
    try:
        for index, record in enumerate(runner.generate(examples, run_id=BENCH_NAME, seed=SEED), start=1):
            payload = record.model_dump(mode='json')
            records.append(payload)
            now = time.perf_counter()
            elapsed = now - started
            avg = elapsed / index
            remaining = max(len(examples) - index, 0) * avg
            forced = sum(1 for row in records if row.get('metadata', {}).get('thinking_force_closed'))
            print(
                f'[{model_name}] {index}/{len(examples)} complete | elapsed {elapsed/60:.1f} min | '
                f'avg {avg:.1f} sec/ex | ETA {remaining/60:.1f} min | forced_close {forced}/{index}',
                flush=True,
            )
            last_print = now
    finally:
        cleanup_vllm_runner(runner)

    total_seconds = time.perf_counter() - started
    scored = score_records(example_rows, records)
    write_jsonl(RUN_ROOT / f'{model_name}_calls.jsonl', records)
    write_jsonl(RUN_ROOT / f'{model_name}_scored_calls.jsonl', scored)
    pd.DataFrame(scored).to_csv(RUN_ROOT / f'{model_name}_scored_calls.csv', index=False)
    summarize_goal1(scored, example_rows).to_csv(RUN_ROOT / f'{model_name}_metrics_summary.csv', index=False)
    metadata = {
        'model_name': model_name,
        'model_id': model_id,
        'max_new_tokens': max_new_tokens,
        'total_seconds': total_seconds,
        'seconds_per_example': total_seconds / len(examples),
        'examples_per_hour': len(examples) * 3600 / total_seconds,
        'examples_per_24h': len(examples) * TARGET_HOURS_PER_CHECKPOINT * 3600 / total_seconds,
    }
    write_json(RUN_ROOT / f'{model_name}_runtime.json', metadata)
    print(f'Wrote {model_name} artifacts to {RUN_ROOT}')
    return records, scored, metadata


## 8. Run 32B Instruct

This run uses `max_new_tokens=2048`, sampling, and the same forced-close machinery. It should rarely force-close unless the model emits an unclosed `<think>` block and hits the first-phase cap.


In [ ]:
all_records = {}
all_scored = {}
runtime_rows = []

if RUN_INSTRUCT:
    records, scored, runtime = run_one_model('olmo31_32b_instruct', INSTRUCT_MODEL_ID, INSTRUCT_MAX_NEW_TOKENS)
    all_records['olmo31_32b_instruct'] = records
    all_scored['olmo31_32b_instruct'] = scored
    runtime_rows.append(runtime)
else:
    print('Skipping Instruct run.')


## 9. Run 32B Think

This run gives the Think phase 4096 tokens, then reserves 100 final-answer tokens for the forced continuation path. If vLLM does not release memory after the Instruct run, restart the runtime, set `RUN_INSTRUCT = False`, and rerun from setup using the same configuration.


In [ ]:
if RUN_THINK:
    records, scored, runtime = run_one_model('olmo31_32b_think', THINK_MODEL_ID, THINK_MAX_NEW_TOKENS)
    all_records['olmo31_32b_think'] = records
    all_scored['olmo31_32b_think'] = scored
    runtime_rows.append(runtime)
else:
    print('Skipping Think run.')


## 10. Summarize Runtime, Accuracy, And Termination

Use this table to decide the full-run subset size. The paired recommendation uses the slower measured checkpoint so both checkpoints fit in the 24-hour target.


In [ ]:
def summarize_scored(model_name: str, scored: list[dict], runtime: dict) -> dict:
    df = pd.DataFrame(scored)
    forced = df['metadata'].apply(lambda value: bool((value or {}).get('thinking_force_closed')))
    return {
        'model_name': model_name,
        'examples': len(df),
        'total_minutes': runtime['total_seconds'] / 60,
        'seconds_per_example': runtime['seconds_per_example'],
        'examples_per_hour': runtime['examples_per_hour'],
        'examples_per_24h': runtime['examples_per_24h'],
        'examples_per_cell_24h': math.floor(runtime['examples_per_24h'] / num_cells),
        'parsed_accuracy': df['parsed_answer_correct'].mean(),
        'output_format_accuracy': df['parsed_output_format_correct'].mean(),
        'forced_close_rate': forced.mean(),
        'avg_output_tokens': df['token_count_output'].mean(),
        'p50_output_tokens': df['token_count_output'].quantile(0.5),
        'p90_output_tokens': df['token_count_output'].quantile(0.9),
    }

runtime_by_name = {row['model_name']: row for row in runtime_rows}
summary_rows = [summarize_scored(name, scored, runtime_by_name[name]) for name, scored in all_scored.items()]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(RUN_ROOT / 'runtime_accuracy_termination_summary.csv', index=False)

if not summary_df.empty:
    paired_examples_per_cell = int(summary_df['examples_per_cell_24h'].min())
    paired_total = paired_examples_per_cell * num_cells
    print(f'Measured cells: {num_cells}')
    print(f'Recommended paired full-run size: {paired_examples_per_cell} examples per cell, {paired_total} total examples per checkpoint.')
    print('This is based on the slower measured checkpoint so each checkpoint is estimated to finish within the target budget.')


## 11. Cell-Level Coverage

This shows whether particular digit lengths, slices, or prompt modes are unusually slow or have high force-close rates. Use it to avoid over-allocating examples to pathological cells.


In [ ]:
example_df = pd.DataFrame(example_rows)
cell_summaries = []
for model_name, scored in all_scored.items():
    scored_df = pd.DataFrame(scored)
    scored_df['forced_close'] = scored_df['metadata'].apply(lambda value: bool((value or {}).get('thinking_force_closed')))
    merged = scored_df.merge(
        example_df[['id', 'n_digits', 'slice_name', 'prompt_mode', 'a', 'b', 'answer']],
        left_on='example_id',
        right_on='id',
        how='left',
    )
    grouped = merged.groupby(['n_digits', 'slice_name', 'prompt_mode'], dropna=False).agg(
        examples=('example_id', 'count'),
        parsed_accuracy=('parsed_answer_correct', 'mean'),
        output_format_accuracy=('parsed_output_format_correct', 'mean'),
        forced_close_rate=('forced_close', 'mean'),
        avg_output_tokens=('token_count_output', 'mean'),
        avg_latency_seconds=('latency_seconds', 'mean'),
    ).reset_index()
    grouped.insert(0, 'model_name', model_name)
    cell_summaries.append(grouped)

if cell_summaries:
    cell_summary_df = pd.concat(cell_summaries, ignore_index=True)
    display(cell_summary_df)
    cell_summary_df.to_csv(RUN_ROOT / 'cell_level_summary.csv', index=False)
else:
    print('No completed runs to summarize.')


## 12. Inspect Outputs

Print full outputs for a small number of examples. The full JSONL/CSV artifacts are saved in `RUN_ROOT`.


In [ ]:
MAX_PRINT = 4
for model_name, scored in all_scored.items():
    print('\n' + '=' * 120)
    print(model_name)
    for row in scored[:MAX_PRINT]:
        example = next(item for item in example_rows if item['id'] == row['example_id'])
        print('\n' + '-' * 120)
        print('example:', {key: example[key] for key in ['n_digits', 'slice_name', 'prompt_mode', 'a', 'b', 'answer']})
        print('metadata:', row.get('metadata'))
        print('token_count_output:', row.get('token_count_output'))
        print('parsed_answer_correct:', row.get('parsed_answer_correct'))
        print('prompt:', row.get('prompt'))
        print('decoded_output:')
        print(row.get('decoded_output'))

print('\nArtifacts saved under:', RUN_ROOT)
